# [2장 3강] - 실습: 연립방정식과 행렬 해법

**실습 목표**
- 문장으로 주어진 조건을 연립방정식으로 정리하고 `Ax = b` 형태로 바꿀 수 있다.
- `np.linalg.solve`로 선형시스템을 풀고 검산할 수 있다.
- rank 비교로 해가 유일해·무한해·불능 중 어느 경우인지 판단할 수 있다.
- 과잉결정 문제에서 정규방정식과 최소제곱해로 회귀계수를 구할 수 있다.

- `Ax = b` 세우기 → 해 구하기와 검산 → 해의 3가지 경우 판별 → 과잉결정과 최소제곱 순서로 진행합니다.
- 개인 실습으로 진행하며, 해를 구한 뒤에는 반드시 `A @ x`를 계산해 `b`와 맞는지 검산합니다.

**실습에 필요한 데이터셋/파일**
- 분야: 제조 품질
- 데이터셋: UCI Wine Quality
- 사용 방식: `ucimlrepo.fetch_ucirepo(id=186)`
- 출처: https://archive.ics.uci.edu/dataset/186/wine+quality
- 사용 목적: 필수 1~2는 소규모 배합 문제로, 심화 1은 Wine Quality 데이터의 품질 점수 회귀로 진행합니다.
- 준비물: Python, NumPy, pandas, scikit-learn, ucimlrepo

- Wine Quality의 타깃(`quality`)은 DataFrame으로 반환될 수 있으므로 **숫자형 Series로 변환**한 뒤 회귀에 사용합니다.
- `np.linalg.solve`는 **정사각행렬이면서 해가 유일할 때만** 사용할 수 있습니다. 그 외에는 `LinAlgError`가 발생하므로, 해의 종류를 먼저 판별한 뒤 적절한 함수를 선택합니다.

In [2]:
# 최초 1회만 실행 (새 환경일 때)
# !pip -q install ucimlrepo scikit-learn pandas numpy

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def load_uci(dataset_id):
    """UCI에서 데이터를 불러오고, 실패하면 구조가 비슷한 대체 데이터를 사용합니다."""
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=dataset_id)
        X, y = ds.data.features.copy(), ds.data.targets.copy()
        if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
            y = y.iloc[:, 0]
        return X, y
    except Exception as e:
        print('[안내] UCI 로드 실패:', e)
        print('[안내] 대체 데이터(sklearn wine)로 진행합니다.')
        from sklearn.datasets import load_wine
        data = load_wine(as_frame=True)
        return data.data, data.target


def numeric_frame(X):
    """수치형 컬럼만 남기고 결측값을 중앙값으로 채웁니다."""
    Xn = X.select_dtypes(include='number').copy()
    Xn = Xn.replace([np.inf, -np.inf], np.nan)
    return Xn.fillna(Xn.median(numeric_only=True))


def describe_solution(A, b):
    """rank 비교로 해의 종류를 판별합니다."""
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float).reshape(-1)
    rank_A = np.linalg.matrix_rank(A)
    rank_Ab = np.linalg.matrix_rank(np.column_stack([A, b]))
    n_vars = A.shape[1]
    if rank_A < rank_Ab:
        kind = '해 없음(불능)'
    elif rank_A == n_vars:
        kind = '유일해'
    else:
        kind = '무한해'
    return {'rank(A)': rank_A, 'rank([A|b])': rank_Ab, '변수 수': n_vars, '판정': kind}


X_raw, y_raw = load_uci(186)   # Wine Quality
print('데이터 shape:', X_raw.shape)

데이터 shape: (6497, 11)


## 필수 1 : 배합 조건을 방정식으로 바꿔 한 번에 풀기
와인 블렌딩 담당자는 원액 두 종류를 섞어 목표 알코올 도수와 목표 산도를 동시에 맞춰야 합니다. "조건 2개, 미지수 2개"인 이 문제는 손으로도 풀 수 있지만, 조건이 늘어나면 계산이 급격히 복잡해집니다. 조건을 행렬 A와 벡터 b로 정리해 Ax = b 한 줄로 표현하고, NumPy로 한 번에 푸는 방법을 익힙니다.

### 문제 1-1 : 조건을 Ax = b로 정리하고 풀기
1. 다음 조건을 연립방정식으로 정리합니다.
    - 원액 A 1L당 알코올 기여 2, 원액 B 1L당 알코올 기여 1 → 목표 알코올 기여 합계 8
    - 원액 A 1L당 산도 기여 1, 원액 B 1L당 산도 기여 3 → 목표 산도 기여 합계 13
2. 계수 행렬 `A`와 상수항 벡터 `b`를 만들고 shape을 확인합니다.
3. `np.linalg.solve(A, b)`로 해를 구합니다.
4. `A @ x`를 계산해 `b`와 일치하는지 검산합니다.
5. 구한 해를 "원액 A ○L, 원액 B ○L를 섞으면 됩니다" 형식의 문장으로 작성합니다.

In [3]:
# 1. 다음 조건을 연립방정식으로 정리합니다.
#     - 원액 A 1L당 알코올 기여 2, 원액 B 1L당 알코올 기여 1 → 목표 알코올 기여 합계 8
#     - 원액 A 1L당 산도 기여 1, 원액 B 1L당 산도 기여 3 → 목표 산도 기여 합계 13
# 알코올:  2x + 1y = 8      (A는 1L당 2, B는 1L당 1, 합쳐서 8)
# 산도:    1x + 3y = 13     (A는 1L당 1, B는 1L당 3, 합쳐서 13)

# 2. 계수 행렬 `A`와 상수항 벡터 `b`를 만들고 shape을 확인합니다.
A = np.array([
    [2, 1],
    [1, 3]
    ])
b = np.array([8, 13])
print(A.shape) # (2, 2)
print(b.shape) # (2,)

# 3. `np.linalg.solve(A, b)`로 해를 구합니다.
print(np.linalg.solve(A, b)) # [2.2 3.6]

# 4. `A @ x`를 계산해 `b`와 일치하는지 검산합니다.
x = np.linalg.solve(A, b)
print(f"A @ x = {A @ x}")

# 5. 구한 해를 "원액 A ○L, 원액 B ○L를 섞으면 됩니다" 형식의 문장으로 작성합니다.
# 원액 A 2.2L, 원액 B 3.6L 섞으면 됩니다.

(2, 2)
(2,)
[2.2 3.6]
A @ x = [ 8. 13.]


### 문제 1-2 : 역행렬 해법과 solve 비교하기
1. `np.linalg.inv(A) @ b`로 해를 구해 `solve` 결과와 비교합니다.
2. 두 결과의 차이를 노름으로 확인합니다.
3. 조건을 살짝 바꿔 거의 종속에 가까운 행렬 `A_ill = [[2, 1], [2.000001, 1.0000004]]`와 상수항 `b_ill = [8, 8.000002]`를 만들고 조건수(`np.linalg.cond`)를 계산합니다. (2행이 1행의 정확한 배수가 되면 특이행렬이 되어 이후 단계를 진행할 수 없으니 숫자를 그대로 사용하세요.)
4. `A_ill`에 대해 두 방식으로 해를 구해 결과가 얼마나 벌어지는지 확인하고, 이어서 `b_ill`의 둘째 원소를 `0.0000001`만 바꿔 다시 풀어 해가 얼마나 움직이는지 보여 줍니다.
5. 실무에서 역행렬을 직접 구하기보다 `solve`를 권장하는 이유를 한 문장으로 작성합니다.

In [4]:
# 1. `np.linalg.inv(A) @ b`로 해를 구해 `solve` 결과와 비교합니다.
A_inv = np.linalg.inv(A)
A_inv_b = A_inv @ b
solve = np.linalg.solve(A, b)
# print(f"np.linalg.inv(A) = \n{A_inv}\n")
print(f"A_inv @ b = {A_inv_b}")
print(f"solve(A, b) = {solve}")

# 2. 두 결과의 차이를 노름으로 확인합니다.
print(f"A_inv norm: {np.linalg.norm(A_inv_b)}")
print(f"solve norm: {np.linalg.norm(solve)}")
print(f"Two norm is same? {np.allclose(np.linalg.norm(A_inv_b), np.linalg.norm(solve))}")

# 3. 조건을 살짝 바꿔 거의 종속에 가까운 행렬 `A_ill = [[2, 1], [2.000001, 1.0000004]]`와 상수항 `b_ill = [8, 8.000002]`를 만들고 조건수(`np.linalg.cond`)를 계산합니다. (2행이 1행의 정확한 배수가 되면 특이행렬이 되어 이후 단계를 진행할 수 없으니 숫자를 그대로 사용하세요.)
A_ill = np.array([
    [2, 1],
    [2.000001, 1.0000004]
    ])

b_ill = np.array([8, 8.000002])

print(f"A_ill Condition: {np.linalg.cond(A_ill)}\n")

# 4. `A_ill`에 대해 두 방식으로 해를 구해 결과가 얼마나 벌어지는지 확인하고, 이어서 `b_ill`의 둘째 원소를 `0.0000001`만 바꿔 다시 풀어 해가 얼마나 움직이는지 보여 줍니다.
A_ill_solve = np.linalg.solve(A_ill, b_ill)
print(f"A_ill solve = {A_ill_solve}")

A_ill_inv_b = np.linalg.inv(A_ill) @ b_ill
print(f"A_ill_inv @ b = {A_ill_inv_b}")

b_ill_2 = np.array([8, 8.000021])
A_ill_solve_2 = np.linalg.solve(A_ill, b_ill_2)
print(f"A_ill solve 2 = {A_ill_solve_2}")

# 역행렬(직선에 가깝게찌부된 걸 다시 펴야함) 값이 극단적으로 바뀌게 되며, b의 작은 오차에도(A_inv @ b) x의 값, 즉 해가 엄청 크게 흔들린다.
A_ill_inv_b_2 = np.linalg.inv(A_ill) @ b_ill_2
print(f"A_ill_inv 2 @ b = {A_ill_inv_b_2}")

# 5. 실무에서 역행렬을 직접 구하기보다 `solve`를 권장하는 이유를 한 문장으로 작성합니다.
# solve는 역행렬을 명시적으로 구하지 않고 소거법으로 해를 직접 계산하기 때문에 속도가 빠르며 수치 오차가 적다.

A_inv @ b = [2.2 3.6]
solve(A, b) = [2.2 3.6]
A_inv norm: 4.219004621945797
solve norm: 4.219004621945797
Two norm is same? True
A_ill Condition: 50000024.090334095

A_ill solve = [-6. 20.]
A_ill_inv @ b = [-6.00000001 20.00000001]
A_ill solve 2 = [  89.00000001 -170.00000002]
A_ill_inv 2 @ b = [  89.         -170.00000001]


## 필수 2 : 답이 하나가 아닐 수도 있다
블렌딩 조건을 바꿨더니 어떤 배합으로도 목표를 맞출 수 없거나, 반대로 여러 배합이 모두 조건을 만족하는 상황이 생깁니다. solve는 이런 경우 오류를 내거나 잘못된 답을 줍니다. 계산 전에 rank를 비교해 해가 어떤 종류인지 먼저 판단하는 습관을 들입니다.

### 문제 2-1 : rank 비교로 해의 종류 판별하기
1. 다음 세 가지 시스템을 정의합니다.
    - **유일해**: `A1 = [[2, 1], [1, 3]]`, `b1 = [8, 13]`
    - **무한해**: `A2 = [[2, 1], [4, 2]]`, `b2 = [8, 16]` (두 번째 식이 첫 식의 2배)
    - **불능**: `A3 = [[2, 1], [4, 2]]`, `b3 = [8, 20]` (좌변은 배수인데 우변은 아님)
2. 각 시스템에 대해 `rank(A)`, `rank([A|b])`, 변수 수를 계산하고 이론의 판정 기준과 비교합니다.
3. 세 결과를 표로 정리합니다.
4. 각 시스템에 `np.linalg.solve`를 시도해 어떤 결과·오류가 나는지 확인합니다.
5. 세 경우를 두 직선의 위치 관계(한 점에서 만남 / 완전히 겹침 / 평행)로 해석해 작성합니다.

In [ ]:
# 1. 다음 세 가지 시스템을 정의합니다.
#     - **유일해**: `A1 = [[2, 1], [1, 3]]`, `b1 = [8, 13]`
#     - **무한해**: `A2 = [[2, 1], [4, 2]]`, `b2 = [8, 16]` (두 번째 식이 첫 식의 2배)
#     - **불능**: `A3 = [[2, 1], [4, 2]]`, `b3 = [8, 20]` (좌변은 배수인데 우변은 아님)
A1 = np.array([
    [2, 1],
    [1, 3]
])
b1 = np.array([8, 13])

A2 = np.array([
    [2, 1],
    [4, 2]
])
b2 = np.array([8, 16])

A3 = np.array([
    [2, 1],
    [4, 2]
])
b3 = np.array([8, 20])

# 2. 각 시스템에 대해 `rank(A)`, `rank([A|b])`, 변수 수를 계산하고 이론의 판정 기준과 비교합니다.
# rank란 서로 선형독립이 되도록 고를 수 있는 컬럼의 최대 수
A1_rank = np.linalg.matrix_rank(A1)
A1b1_rank = np.linalg.matrix_rank(np.column_stack([A1, b1]))
print(f"A1 rank: {A1_rank}, A|B rank: {A1b1_rank}")

A2_rank = np.linalg.matrix_rank(A2)
A2b2_rank = np.linalg.matrix_rank(np.column_stack([A2, b2]))
print(f"A2 rank: {A2_rank}, A|B rank: {A2b2_rank}")

A3_rank = np.linalg.matrix_rank(A3)
A3b3_rank = np.linalg.matrix_rank(np.column_stack([A3, b3]))
print(f"A3 rank: {A3_rank}, A|B rank: {A3b3_rank}")

# 1단계 — 해가 존재하는가
    # r = r'  →  해가 존재함 (consistent)
    # r < r'  →  해가 없음 (불능, inconsistent)

# 2단계 — 해가 존재할 때, 몇 개인가
    # r = n  →  유일해 (unique solution)
    # r < n  →  무한해 (자유변수 n − r 개)

for A, b in ((A1, b1), (A2, b2), (A3, b3)):
    A_rank = np.linalg.matrix_rank(A)
    Ab_rank = np.linalg.matrix_rank(np.column_stack([A, b]))
    print(f"A: {A}\nb: {b}")
    if A_rank == Ab_rank: # 해가 존재함
        if A_rank == A.shape[1]:
            print("유일해가 존재함\n")
        else:
            print("무한해가 존재함\n")
    else:
        print("불능\n")

# 3. 세 결과를 표로 정리합니다.

# 4. 각 시스템에 `np.linalg.solve`를 시도해 어떤 결과·오류가 나는지 확인합니다.
for A, b, name in ((A1, b1, "A1"), (A2, b2, "A2"), (A3, b3, "A3")):
    try:
        A_solve = np.linalg.solve(A, b)
        print(f"{name}: {A_solve}")
    except Exception as e:
        print(f"{name}: {e}")

# 5. 세 경우를 두 직선의 위치 관계(한 점에서 만남 / 완전히 겹침 / 평행)로 해석해 작성합니다.


A1 rank: 2, A|B rank: 2
A2 rank: 1, A|B rank: 1
A3 rank: 1, A|B rank: 2
A: [[2 1]
 [1 3]]
b: [ 8 13]
유일해가 존재함

A: [[2 1]
 [4 2]]
b: [ 8 16]
무한해가 존재함

A: [[2 1]
 [4 2]]
b: [ 8 20]
불능

A1: [2.2 3.6]
A2: Singular matrix
A3: Singular matrix


## 심화 1 : 조건이 미지수보다 훨씬 많을 때 - 최소제곱과 정규방정식
품질 분석팀은 이화학 측정값으로 와인 품질 점수를 예측하는 회귀식을 만들려 합니다. 이때 방정식(샘플)이 수천 개, 미지수(회귀계수)는 10여 개라 모든 샘플을 정확히 만족하는 해는 존재하지 않습니다. 이런 과잉결정 상황에서 오차 제곱합을 최소화하는 해를 구하는 방법을 확인합니다.

### 문제 3-1 : 최소제곱해와 정규방정식 비교하기
1. Wine Quality 데이터를 표준화하고 앞에 1로 채운 절편 컬럼을 붙여 설계행렬 `Xd`를 만듭니다.
2. `Xd`의 shape을 확인해 방정식 수와 미지수 수를 비교하고, `describe_solution()`으로 `Xd x = y`의 해 종류를 판정합니다.
3. `np.linalg.lstsq`로 최소제곱해 `coef_lstsq`를 구하고 RMSE를 계산합니다.
4. 정규방정식 `XᵀXβ = Xᵀy`를 `np.linalg.solve`로 풀어 `coef_normal`을 구합니다.
5. 두 계수의 차이와 `XᵀX`의 조건수를 확인합니다.
6. 정확한 해가 없는 문제에서도 최소제곱해는 항상 구할 수 있는 이유, 그리고 실무에서 정규방정식보다 `lstsq`를 권장하는 이유를 2~3문장으로 작성합니다.

In [ ]:
# 1. Wine Quality 데이터를 표준화하고 앞에 1로 채운 절편 컬럼을 붙여 설계행렬 `Xd`를 만듭니다.
X_num = numeric_frame(X_raw)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)
ones = np.ones((X_scaled.shape[0], 1))
Xd = np.hstack([ones, X_scaled])
Xd

# 2. `Xd`의 shape을 확인해 방정식 수와 미지수 수를 비교하고, `describe_solution()`으로 `Xd x = y`의 해 종류를 판정합니다.
print(f"Xd의 방정식 수: {Xd.shape[0]}, \nXd의 미지수 수: {Xd.shape[1]}\n")
print(f"Xd의 rank 수: {np.linalg.matrix_rank(Xd)}, \nXd|y의 rank 수: {np.linalg.matrix_rank(np.column_stack([Xd, y_raw]))}")
print("Xd의 rank < Xd|y의 rank => 불능")

# 3. `np.linalg.lstsq`로 최소제곱해 `coef_lstsq`를 구하고 RMSE를 계산합니다.
coef_lstsq = np.linalg.lstsq(Xd, y_raw, rcond=None)[0]
coef_lstsq
y_pred = Xd @ coef_lstsq
rmse = np.sqrt(mean_squared_error(y_raw, y_pred))
print(f"RMSE: {rmse}")

# 4. 정규방정식 `XᵀXβ = Xᵀy`를 `np.linalg.solve`로 풀어 `coef_normal`을 구합니다.
XtX = Xd.T @ Xd
Xty = Xd.T @ y_raw
coef_normal = np.linalg.solve(XtX, Xty)
coef_normal

# 5. 두 계수의 차이와 `XᵀX`의 조건수를 확인합니다.
print(f"coef_lstsq: {coef_lstsq}, \ncoef_normal: {coef_normal}")
print(np.allclose(coef_normal, coef_lstsq))

# 6. 정확한 해가 없는 문제에서도 최소제곱해는 항상 구할 수 있는 이유, 그리고 실무에서 정규방정식보다 `lstsq`를 권장하는 이유를 2~3문장으로 작성합니다.
# 정확히 만족하는 해는 없지만, 관측값과 예측값 사이의 오차를 최소화하게 해주면 답이 있다.
# 정규방정식은 XᵀX를 만드는데 이 과정에서 행렬의 조건수가 제곱이 되며, 오차가 증폭 돼 계수가 부정확해질 수 있다.

Xd의 방정식 수: 6497, 
Xd의 미지수 수: 12

Xd의 rank 수: 12, 
Xd|y의 rank 수: 13
Xd의 rank < Xd|y의 rank => 불능
RMSE: 0.7346532973298834
coef_lstsq: [ 5.81837771  0.08774096 -0.21860267 -0.01593384  0.20722804 -0.01694492
  0.10595378 -0.14023679 -0.1648152   0.07062775  0.11431158  0.31846532], 
coef_normal: [ 5.81837771  0.08774096 -0.21860267 -0.01593384  0.20722804 -0.01694492
  0.10595378 -0.14023679 -0.1648152   0.07062775  0.11431158  0.31846532]
True
